# Discrete vs Continuous Rewards in scRL

This notebook demonstrates the correct usage of the two reward functions in scRL, along with the **Decision** and **Contribution** modes.

## Naming Philosophy

scRL provides two reward functions based on the **type of input features**:

| Function | Feature Type | Description | Input Parameters |
|----------|--------------|-------------|------------------|
| `d_rewards` | **Discrete** | Rewards based on categorical cluster/lineage membership | `starts` and `ends` are **cluster names** |
| `c_rewards` | **Continuous** | Rewards based on continuous gene expression values | `reward_keys` are **gene names** from projected data |

- **`d`** = **discrete** features (cluster labels, lineage identities)
- **`c`** = **continuous** features (gene expression levels, numerical values)

## Reward Modes

Both functions support two modes that determine the reward pattern along pseudotime:
- **Decision**: Higher rewards at earlier pseudotime (identifies early fate decision points)
- **Contribution**: Higher rewards at later pseudotime (identifies late fate contributions)

In [ ]:
import scRL
import scanpy as sc
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

sc.set_figure_params(figsize=(5,5), frameon=False, fontsize=14)

## Load and Preprocess Data

In [ ]:
# Load example dataset
adata = sc.datasets.paul15()

# Standard preprocessing
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50)
sc.tl.tsne(adata, perplexity=50)
sc.pp.neighbors(adata)
sc.tl.leiden(adata)

sc.pl.tsne(adata, color='leiden', legend_loc='on data', s=100, title='Leiden Clusters')

## Create Grid Embedding

In [ ]:
# Extract embedding and cluster information
X = adata.obsm['X_tsne']
X_pca = adata.obsm['X_pca']
clusters = adata.obs['leiden']
cluster_colors = adata.uns['leiden_colors']

# Create grid embedding
gres = scRL.grids_from_embedding(X)

# Project cluster information
scRL.project_cluster(gres, clusters, cluster_colors)

# Align pseudotime (cluster '4' as starting point)
scRL.align_pseudotime(gres, '4')
scRL.project_back(gres, 'pseudotime')

## Method 1: Discrete Rewards with `d_rewards`

Use `d_rewards` when your target features are **discrete/categorical** (e.g., cluster membership, lineage identity).

The agent receives rewards when reaching grid points belonging to specified terminal clusters.

In [ ]:
# d_rewards for lineage-based analysis
# starts: starting cluster(s)
# ends: terminal cluster(s)

# Decision mode - rewards early fate decisions
scRL.d_rewards(gres, starts=['4'], ends=['0', '3', '5'], mode='Decision')
print("Decision mode rewards generated successfully!")

In [ ]:
# Contribution mode - rewards late fate contributions
scRL.d_rewards(gres, starts=['4'], ends=['0', '3', '5'], mode='Contribution')
print("Contribution mode rewards generated successfully!")

### Common Mistake

**Do NOT use `c_rewards` with cluster names.**

```python
# This will cause KeyError
scRL.c_rewards(gres, ['5'], ['0','6','3'], mode='Contribution')
```

The error occurs because:
- `c_rewards` is designed for **continuous** features (gene expression values)
- Cluster names like `'5'`, `'0'`, `'6'`, `'3'` are **discrete** categorical labels
- For discrete cluster-based analysis, use `d_rewards` instead

## Method 2: Continuous Rewards with `c_rewards`

Use `c_rewards` when your target features are **continuous/numerical** (e.g., gene expression levels).

The agent receives rewards proportional to the continuous values at each grid point.

### Step 1: Project Continuous Data onto Grid

In [ ]:
# First, you MUST project gene expression data onto the grid
# Select genes of interest (e.g., lineage markers)
gene_names = ['Mpo', 'Klf1']  # Example: Mpo (myeloid), Klf1 (erythroid)

# Create gene expression dataframe
exp = pd.DataFrame(adata[:, gene_names].X, columns=gene_names)

# Project gene expression onto grid
scRL.project(gres, exp)

# Now check what columns are available
print("Available gene columns in gres.grids['proj']:")
print(list(gres.grids['proj'].columns))

### Step 2: Use `c_rewards` with Gene Names

In [ ]:
# Use gene names from projected data
# reward_keys: genes whose high expression indicates the target fate
# starts: starting cluster(s) for the agent

scRL.c_rewards(gres, 
               reward_keys=['Mpo'],  # Gene names, NOT cluster names!
               starts=['4'],          # Starting cluster
               mode='Decision')
print("Gene-based Decision mode rewards generated successfully!")

In [ ]:
# Contribution mode for gene-based analysis
scRL.c_rewards(gres,
               reward_keys=['Klf1'],  # Gene names
               starts=['4'],
               mode='Contribution')
print("Gene-based Contribution mode rewards generated successfully!")

## Summary: Which Function to Use?

| Feature Type | Function | Use When | Example |
|--------------|----------|----------|---------|
| **Discrete** | `d_rewards` | Analyzing fate decisions to specific clusters/lineages | `scRL.d_rewards(gres, starts=['HSC'], ends=['Ery', 'Mono'])` |
| **Continuous** | `c_rewards` | Analyzing fate decisions based on gene expression values | `scRL.c_rewards(gres, reward_keys=['Gata1'], starts=['HSC'])` |

### Key Points:
1. **`d_rewards`**: For **discrete** features (cluster names, lineage labels). Uses cluster names directly.
2. **`c_rewards`**: For **continuous** features (gene expression). Requires `scRL.project(gres, exp)` first, then uses column names from projected data.
3. Both functions support `mode='Decision'` (early rewards) and `mode='Contribution'` (late rewards).

## Training Example with Contribution Mode

In [ ]:
# Generate rewards with Contribution mode for lineage analysis
scRL.d_rewards(gres, starts=['4'], ends=['0', '3', '5'], mode='Contribution')

# Train the model
t = scRL.trainer('ActorCritic', gres, 
                 reward_type='d',        # 'd' for d_rewards, 'c' for c_rewards
                 reward_mode='Contribution',
                 X_latent = X_pca, 
                 starts_prob = False,
                 num_episodes=1000)

r_l, v_l = t.train()

In [ ]:
# Get state values
scRL.get_state_value(gres, t, 'contribution_value')
scRL.project_back(gres, 'contribution_value')

# Visualize
adata.obs['contribution_value'] = gres.embedding['contribution_value']
sc.pl.tsne(adata, color='contribution_value', cmap='plasma', 
           title='Lineage Contribution Value', s=100)